In [1]:
# Cell 1
!pip install -q torchaudio wandb scikit-learn

import os
import random
import glob
import torch
import torchaudio
import wandb
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from tqdm.notebook import tqdm

# Kaggle dataset path (Adjust if your dataset folder is named differently)
BASE_DIR = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup" 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Cell 2: WandB Login
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("wandb_api")
    wandb.login(key=wandb_api_key)
    print("✅ Successfully logged into Weights & Biases!")
except Exception as e:
    print(f"Secrets not set up. Please log in manually. Error: {e}")
    wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2002444 (23f2002444-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Successfully logged into Weights & Biases!


In [3]:
# Cell 3: Dataloader with SpecAugment
class DynamicMashupDataset(Dataset):
    def __init__(self, base_dir, split='train', duration_sec=6.0, target_sr=22050, epoch_length=4000):
        self.base_dir = base_dir
        self.target_sr = target_sr
        self.num_frames = int(duration_sec * target_sr)
        self.epoch_length = epoch_length
        self.split = split
        
        self.genres = sorted(["blues", "classical", "country", "disco", "hiphop", 
                              "jazz", "metal", "pop", "reggae", "rock"])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
        
        self.stems_dir = os.path.join(base_dir, "genres_stems")
        self.songs_by_genre = {g: [] for g in self.genres}
        
        for genre in self.genres:
            genre_path = os.path.join(self.stems_dir, genre)
            if os.path.exists(genre_path):
                songs = [os.path.join(genre_path, d) for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))]
                split_idx = int(len(songs) * 0.8) 
                self.songs_by_genre[genre] = songs[:split_idx] if split == 'train' else songs[split_idx:]
            
        self.noise_files = glob.glob(os.path.join(base_dir, "ESC-50-master", "audio", "*.wav"))
        
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.target_sr, n_mels=128, n_fft=1024, hop_length=512
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        
        # 🚀 NEW: SpecAugment Initializations
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=35)

    def __len__(self):
        return self.epoch_length

    def load_and_process_audio(self, filepath):
        try:
            waveform, orig_sr = torchaudio.load(filepath)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            if orig_sr != self.target_sr:
                resampler = torchaudio.transforms.Resample(orig_sr, self.target_sr)
                waveform = resampler(waveform)
            if waveform.shape[1] > self.num_frames:
                max_start = waveform.shape[1] - self.num_frames
                start_frame = random.randint(0, max_start) if self.split == 'train' else 0
                waveform = waveform[:, start_frame:start_frame + self.num_frames]
            elif waveform.shape[1] < self.num_frames:
                waveform = torch.nn.functional.pad(waveform, (0, self.num_frames - waveform.shape[1]))
            return waveform
        except Exception as e:
            return torch.zeros((1, self.num_frames))

    def __getitem__(self, idx):
        genre = random.choice(self.genres)
        label = self.genre_to_idx[genre]
        
        if not self.songs_by_genre[genre]:
             return torch.zeros((1, 128, 259)), torch.tensor(label, dtype=torch.long)
             
        num_songs_to_pick = min(4, len(self.songs_by_genre[genre]))
        selected_songs = random.sample(self.songs_by_genre[genre], num_songs_to_pick)
        
        stems = ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']
        mixed_waveform = torch.zeros((1, self.num_frames))
        
        for i, song_path in enumerate(selected_songs):
            stem_name = stems[i % len(stems)]
            stem_path = os.path.join(song_path, stem_name)
            if not os.path.exists(stem_path) and stem_name == 'other.wav':
                stem_path = os.path.join(song_path, 'others.wav')

            if os.path.exists(stem_path):
                stem_wave = self.load_and_process_audio(stem_path)
                mixed_waveform += (stem_wave * random.uniform(0.5, 1.2))
                
        if self.noise_files:
            noise_path = random.choice(self.noise_files)
            noise_wave = self.load_and_process_audio(noise_path)
            mixed_waveform += (noise_wave * random.uniform(0.05, 0.25))
        
        mixed_waveform = mixed_waveform / (torch.max(torch.abs(mixed_waveform)) + 1e-6)
        
        mel_spec = self.mel_spectrogram(mixed_waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)
        
        # 🚀 NEW: Apply SpecAugment ONLY during training
        if self.split == 'train':
            mel_spec_db = self.freq_mask(mel_spec_db)
            mel_spec_db = self.time_mask(mel_spec_db)
        
        return mel_spec_db, torch.tensor(label, dtype=torch.long)

In [4]:
# Cell 4: Custom Model Architecture (From Scratch)
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

class AudioResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AudioResNet, self).__init__()
        self.in_channels = 32
        
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(32, 2, stride=1)
        self.layer2 = self._make_layer(64, 2, stride=2)
        self.layer3 = self._make_layer(128, 2, stride=2)
        self.layer4 = self._make_layer(256, 2, stride=2)
        
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(ResidualBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model1 = AudioResNet(num_classes=10).to(device)
print("✅ Custom ResNet Model initialized.")

✅ Custom ResNet Model initialized.


In [5]:
# Cell 5: Training Loop for Model 1
def train_model1():
    wandb.init(
        project="messy-mashup-dlgenai",
        name="Model1-CustomResNet-2nd_run",
        config={
            "epochs": 45, 
            "batch_size": 32,
            "learning_rate": 0.001,
            "architecture": "Custom ResNet (From Scratch)",
        }
    )
    config = wandb.config

    train_dataset = DynamicMashupDataset(BASE_DIR, split='train', epoch_length=4000)
    val_dataset = DynamicMashupDataset(BASE_DIR, split='val', epoch_length=800)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model1.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_f1 = 0.0
    
    for epoch in range(config.epochs):
        model1.train()
        running_loss = 0.0
        
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Train]"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model1(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        
        model1.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} [Val]"):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model1(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")
        
        scheduler.step(val_f1)
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1,
            "learning_rate": current_lr
        })
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model1.state_dict(), "best_cnn_model.pth")
            print("🌟 New best model saved!")

    wandb.finish()

train_model1()

wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260322_153739-9pr4rp6n
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Model1-CustomResNet-2nd_run
wandb: ⭐️ View project at https://wandb.ai/23f2002444-indian-institute-of-technology-madras/messy-mashup-dlgenai
wandb: 🚀 View run at https://wandb.ai/23f2002444-indian-institute-of-technology-madras/messy-mashup-dlgenai/runs/9pr4rp6n


Epoch 1/45 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1/45 [Val]:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1 | LR: 0.001000 | Train Loss: 1.5839 | Val Loss: 1.4191 | Val Acc: 0.5162 | Val Macro F1: 0.4571
🌟 New best model saved!


Epoch 2/45 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

...

Epoch 45/45 [Train]:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 45/45 [Val]:   0%|          | 0/25 [00:00<?, ?it/s]

wandb: updating run metadata


Epoch 45 | LR: 0.000016 | Train Loss: 0.2224 | Val Loss: 0.4258 | Val Acc: 0.8612 | Val Macro F1: 0.8603


wandb: uploading history steps 44-44, summary, console lines 59-59
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
wandb: learning_rate ████████▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train_loss █▆▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  val_accuracy ▁▂▂▃▁▃▂▃▅▆▅▅▆▄▇▆▇█▇█▇▇▇▇█▇▇███████▇███▇█
wandb:      val_loss █▇▇▆█▆▇▆▄▃▄▅▃▇▃▃▂▂▂▁▂▂▂▂▂▃▂▁▁▁▁▁▁▁▁▁▁▁▂▂
wandb:  val_macro_f1 ▁▂▂▄▁▃▂▃▆▇▅▅▆▅▇▆▇█▇█▇▇▇▇█▇▇███████▇███▇█
wandb: 
wandb: Run summary:
wandb:         epoch 45
wandb: learning_rate 2e-05
wandb:    train_loss 0.22243
wandb:  val_accuracy 0.86125
wandb:      val_loss 0.42582
wandb:  val_macro_f1 0.86031
wandb: 
wandb: 🚀 View run Model1-CustomResNet-2nd_run at: https://wandb.ai/23f2002444-indian-institute-of-technology-madras/messy-mashup-dlgenai/runs/9pr4rp6n
wandb: ⭐️ View project at: https://wandb.ai/23f2002444-indian-institute-of-technology-madras/messy-mashup-dlgenai
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) 

In [6]:
# Cell 6: Model 1 Inference
def generate_submission1():
    print("Loading test data and custom model weights...")
    model1.load_state_dict(torch.load("best_cnn_model.pth", map_location=device))
    model1.eval()

    test_csv_path = os.path.join(BASE_DIR, "test.csv")
    if not os.path.exists(test_csv_path):
        test_csv_path = "/kaggle/input/test (1).csv" 
            
    test_df = pd.read_csv(test_csv_path)
    
    mel_spectrogram = torchaudio.transforms.MelSpectrogram(
        sample_rate=22050, n_mels=128, n_fft=1024, hop_length=512
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    
    genres = sorted(["blues", "classical", "country", "disco", "hiphop", 
                     "jazz", "metal", "pop", "reggae", "rock"])
    idx_to_genre = {i: g for i, g in enumerate(genres)}

    predictions = []
    
    print("Running Inference on Test Mashups...")
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row['id']
        filename = f"song{int(file_id):04d}.wav"
        audio_path = os.path.join(BASE_DIR, "mashups", filename)
        
        try:
            waveform, sr = torchaudio.load(audio_path)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sr != 22050:
                resampler = torchaudio.transforms.Resample(sr, 22050)
                waveform = resampler(waveform)
                
            mel_spec = mel_spectrogram(waveform)
            mel_spec_db = amplitude_to_db(mel_spec)
            mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)
            
            input_tensor = mel_spec_db.unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model1(input_tensor)
                _, pred_idx = torch.max(output, 1)
                predicted_genre = idx_to_genre[pred_idx.item()]
                
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            predicted_genre = "pop" 
            
        formatted_id = f"{int(file_id):04d}"
        predictions.append({'id': formatted_id, 'genre': predicted_genre})
        
    submission_df = pd.DataFrame(predictions)
    submission_df.to_csv("submission_model1.csv", index=False)
    print("✅ submission_model1.csv created successfully!")

generate_submission1()

Loading test data and custom model weights...
Running Inference on Test Mashups...


  0%|          | 0/3020 [00:00<?, ?it/s]

✅ submission_model1.csv created successfully!
